# MILK10k — Step 2: Unimodal baselines + per-fold embeddings
Produces, for every fold, three single-modality models (**C** clinical, **D** dermoscopic, **M** metadata) and saves:
* `oof_{C,D,M}.csv` — out-of-fold class probabilities (one row per lesion) → Tier-1 unimodal results
* `emb_{C,D}_fold{f}.npy` — 768-d embeddings of **all 5,240 lesions** from the fold-*f* backbone → fusion / modality-dropout experiments then train in minutes on these

Leakage rule: fold *f*'s backbone never sees fold *f* lesions; the fusion model for fold *f* must only use `emb_*_fold{f}.npy`.

Training skeleton adapted from your ConvNeXtV2 notebook, with four changes: lesion-level folds instead of a random image split; model selection on **macro F1** (not weighted F1, which BCC at 48% would dominate); class-weighted loss **only** (no sampler on top — using both over-corrects); ImageNet normalisation (matches the pretrained weights).

In [ ]:
import os, glob, json, time, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold

# ================= PATHS (auto-detected on Kaggle) =================
def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"not found: {pattern}"
    return hits[0]
SUPP_DIR = os.path.dirname(find_one("/kaggle/input/**/supplements/training_gt.csv"))
IMG_DIR  = os.path.dirname(find_one("/kaggle/input/**/images/ISIC_*.jpg"))
OUT      = "/kaggle/working/stage2"; os.makedirs(OUT, exist_ok=True)
print("supplements:", SUPP_DIR); print("images:", IMG_DIR, len(os.listdir(IMG_DIR)), "files")

# ================= CONFIG =================
BACKBONE  = "facebook/convnextv2-tiny-22k-224"   # needs Internet ON (Settings > Internet)
IMG_SIZE  = 224
EPOCHS, PATIENCE, BATCH, LR = 12, 99, 32, 1e-4   # full schedule always completes; best-val checkpoint kept
FOLDS_TO_RUN = [0, 1, 2, 3, 4]   # run a subset per session if needed
MODS_TO_RUN  = ["D", "C"]
SEED, K_FOLDS = 42, 5
np.random.seed(SEED)

## Folds
Uses `milk10k_lesions_folds.csv` if you attached it; otherwise rebuilds it with exactly the notebook-01 recipe and saves it to /kaggle/working. **Keep that file** — every later notebook must use the same folds.

In [ ]:
cands = glob.glob("/kaggle/input/**/milk10k_lesions_folds.csv", recursive=True)
if cands:
    les = pd.read_csv(cands[0]); print("loaded folds:", cands[0])
else:
    gt  = pd.read_csv(f"{SUPP_DIR}/training_gt.csv")
    inp = pd.read_csv(f"{SUPP_DIR}/training_input.csv")
    CL = [c for c in gt.columns if c != "lesion_id"]
    gt["label"] = gt[CL].idxmax(1); gt["y"] = gt["label"].map({c: i for i, c in enumerate(CL)})
    inp["mod"] = inp["image_type"].map({"clinical: close-up": "C", "dermoscopic": "D"})
    MON = [c for c in inp.columns if c.startswith("MONET_")]
    w = inp.pivot(index="lesion_id", columns="mod", values=["isic_id", "image_manipulation"] + MON)
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    les = (inp[inp["mod"] == "D"][["lesion_id", "age_approx", "sex", "skin_tone_class", "site"]]
           .merge(w.reset_index(), on="lesion_id").merge(gt[["lesion_id", "label", "y"]], on="lesion_id"))
    les["tone_group"] = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5],
                               labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str)
    strata = les["label"] + "|" + les["tone_group"]
    strata = np.where(strata.map(strata.value_counts()) >= K_FOLDS, strata, les["label"])
    les["fold"] = -1
    for f, (_, te) in enumerate(StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED).split(les, strata)):
        les.loc[les.index[te], "fold"] = f
    les.to_csv("/kaggle/working/milk10k_lesions_folds.csv", index=False); print("built + saved folds")

CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']
assert sorted(les["label"].unique()) == CLASSES
K = len(CLASSES)
les["path_C"] = IMG_DIR + "/" + les["isic_id_C"] + ".jpg"
les["path_D"] = IMG_DIR + "/" + les["isic_id_D"] + ".jpg"
missing = [p for p in les["path_C"].tolist() + les["path_D"].tolist() if not os.path.exists(p)]
print(les.shape, "| lesions per fold:", les["fold"].value_counts().sort_index().tolist(), "| missing images:", len(missing))

## Metrics (used by every later notebook)

In [ ]:
def evaluate(y, P):
    """y: int labels (n,), P: probabilities (n,K). Returns the paper's metric set."""
    pred = P.argmax(1)
    out = {
        "bal_acc":  balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
        # official MILK10k benchmark style: each class binarised at >= 0.5
        "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)]),
    }
    try:    out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def fold_table(oof, name):
    rows = [dict(fold=f, **evaluate(d["y"].values, d[CLASSES].values)) for f, d in oof.groupby("fold")]
    t = pd.DataFrame(rows); print(f"\n== {name} ==")
    print(t.round(4).to_string(index=False))
    print("mean ± sd:", {k: f"{t[k].mean():.4f} ± {t[k].std():.4f}" for k in t.columns if k != "fold"})
    return t

## M — metadata-only baseline (CPU, ~1 min)
Age, sex, site, skin tone. MONET scores are **not** included: they are image-derived and belong to their image.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

num, cat = ["age_approx", "skin_tone_class"], ["sex", "site"]
Xm = les[num + cat].copy(); Xm["site"] = Xm["site"].fillna("unknown")
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

oof_M = []
for f in range(5):
    tr, te = les["fold"] != f, les["fold"] == f
    clf = make_pipeline(pre, LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0))
    clf.fit(Xm[tr], les.loc[tr, "y"])
    P = np.zeros((te.sum(), K)); P[:, clf.classes_] = clf.predict_proba(Xm[te])
    oof_M.append(pd.DataFrame(P, columns=CLASSES).assign(lesion_id=les.loc[te, "lesion_id"].values,
                                                          y=les.loc[te, "y"].values, fold=f))
oof_M = pd.concat(oof_M); oof_M.to_csv(f"{OUT}/oof_M.csv", index=False)
tab_M = fold_table(oof_M, "M (metadata only)")

## Image models (GPU)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from transformers import AutoModel
from tqdm.auto import tqdm
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(device)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet (pretrained weights expect this)
TF_TRAIN = {
  "D": T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                  T.RandomRotation(180), T.ColorJitter(0.15, 0.15, 0.1, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)]),
  # clinical close-ups: milder geometry, no vertical flip (surrounding anatomy has an orientation)
  "C": T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                  T.RandomRotation(20), T.ColorJitter(0.15, 0.15, 0.1, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])}
TF_EVAL = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])
# NB: hue jitter kept tiny on purpose — colour carries diagnostic and skin-tone information.

class LesionImages(Dataset):
    def __init__(self, df, mod, tf):
        self.paths, self.y, self.tf = df[f"path_{mod}"].values, df["y"].values, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.y[i]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(BACKBONE)
        self.dim = self.backbone.config.hidden_sizes[-1]          # 768 for convnextv2-tiny
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.dim, K))
    def embed(self, x): return self.backbone(pixel_values=x).pooler_output
    def forward(self, x): return self.head(self.embed(x))

def class_weights(y):
    n = np.bincount(y, minlength=K).astype(float)
    w = np.sqrt(n.max() / np.maximum(n, 1))        # sqrt-inverse frequency: MAL_OTH not weighted 280x
    return torch.tensor(w / w.mean(), dtype=torch.float, device=device)

@torch.no_grad()
def predict(model, df, mod, tta=True):
    dl = DataLoader(LesionImages(df, mod, TF_EVAL), batch_size=64, shuffle=False,
                    num_workers=0 if len(df) < 1000 else 4, pin_memory=True)   # 0 workers for small val set = no log spam
    model.eval(); probs, embs = [], []
    for x, _ in dl:
        x = x.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            e = model.embed(x)
            if tta: e = (e + model.embed(torch.flip(x, dims=[3]))) / 2
            p = model.head(e).float().softmax(1)
        probs.append(p.cpu().numpy()); embs.append(e.float().cpu().numpy())
    return np.concatenate(probs), np.concatenate(embs)

In [ ]:
def run_fold(mod, f):
    trval, test = les[les["fold"] != f], les[les["fold"] == f]
    # inner 10% of the training lesions for early stopping (never the test fold)
    tr, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    dl_tr = DataLoader(LesionImages(tr, mod, TF_TRAIN[mod]), batch_size=BATCH, shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)
    model = Net().to(device)
    crit  = nn.CrossEntropyLoss(weight=class_weights(tr["y"].values), label_smoothing=0.05)
    opt   = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR},
                               {"params": model.head.parameters(),     "lr": LR * 10}], weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR, LR * 10], total_steps=EPOCHS * len(dl_tr), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best, bad, ckpt = -1, 0, f"{OUT}/net_{mod}_fold{f}.pt"
    for ep in range(EPOCHS):
        model.train(); t0, tot = time.time(), 0
        for x, y in tqdm(dl_tr, leave=False, desc=f"{mod} f{f} ep{ep}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        Pv, _ = predict(model, va, mod, tta=False)
        m = evaluate(va["y"].values, Pv)
        print(f"{mod} fold{f} ep{ep:02d} loss {tot/len(dl_tr):.3f} | val macroF1 {m['macro_f1']:.4f} "
              f"balAcc {m['bal_acc']:.4f} | {time.time()-t0:.0f}s")
        if m["macro_f1"] > best: best, bad = m["macro_f1"], 0; torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= PATIENCE: print("early stop"); break
    model.load_state_dict(torch.load(ckpt, map_location=device))
    # 1) out-of-fold probabilities for the test fold
    Pt, _ = predict(model, test, mod)
    pd.DataFrame(Pt, columns=CLASSES).assign(lesion_id=test["lesion_id"].values, y=test["y"].values,
                                             fold=f).to_csv(f"{OUT}/oof_{mod}_fold{f}.csv", index=False)
    # 2) embeddings of ALL lesions from this fold's backbone (row order = folds CSV order)
    _, E = predict(model, les, mod)
    np.save(f"{OUT}/emb_{mod}_fold{f}.npy", E.astype(np.float16))
    del model; torch.cuda.empty_cache()

for mod in MODS_TO_RUN:
    for f in FOLDS_TO_RUN:
        if os.path.exists(f"{OUT}/emb_{mod}_fold{f}.npy"): print("skip", mod, f); continue
        run_fold(mod, f)

## Tier-1 unimodal results

In [ ]:
tables = {"M": tab_M}
for mod in ["C", "D"]:
    parts = [f"{OUT}/oof_{mod}_fold{f}.csv" for f in range(5) if os.path.exists(f"{OUT}/oof_{mod}_fold{f}.csv")]
    if len(parts) == 5:
        oof = pd.concat(map(pd.read_csv, parts)); oof.to_csv(f"{OUT}/oof_{mod}.csv", index=False)
        tables[mod] = fold_table(oof, {"C": "C (clinical only)", "D": "D (dermoscopic only)"}[mod])
summary = pd.DataFrame({m: t.drop(columns="fold").mean() for m, t in tables.items()}).T.round(4)
summary.to_csv(f"{OUT}/tier1_unimodal_summary.csv"); summary

In [ ]:
# Per-class recall — the table reviewers will ask for (MEL / SCCKA recall matters most clinically)
from sklearn.metrics import recall_score
rec = {}
for mod in tables:
    oof = pd.read_csv(f"{OUT}/oof_{mod}.csv")
    rec[mod] = recall_score(oof["y"], oof[CLASSES].values.argmax(1), labels=range(K), average=None, zero_division=0)
pd.DataFrame(rec, index=CLASSES).round(3)